We want to solve the problem
\begin{align}
    -\frac{\mu}{2} \text{div}(\nabla u + \nabla u^T) &= \nabla p \quad &\text{ in } \Omega
    \\
    \text{div}(u) &= 0&\text{ in } \Omega
\end{align}
Where $\Omega$ is a round pipe. At the inflow of the pipe we give some velocity profile $u_{in}$ at the outflow we say $p=0$ (set the pressure to some value) and at the walls we have $u=0$.

In [1]:
import qewton

In [2]:
X = qewton.Variable("x", 3)
N = qewton.Variable("n", 3) # normal vectors
U = qewton.Variable("u", 3) # velocity is vector valued
P = qewton.Variable("p", 1) # pressure is scalar

mu = 0.001

In [3]:
mesh_geo = qewton.geometries.MeshGeometry.load_mesh(
    variable=X, file_path="data/stokes_pipe/u_pipe.msh"
)

inflow_goe = mesh_geo.boundary.get_submesh("Inflow")
outflow_geo = mesh_geo.boundary.get_submesh("Outflow")
side_geo = mesh_geo.boundary.get_submesh("Side")

/home/nick7/pioneer/pioneer-backend/src/qewton/backends/torch/base.py:93: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  return torch.as_tensor(data, dtype=converted_type, device=cls.get_device(device))


In [4]:
volume_sampler = qewton.RandomUniformSampler(mesh_geo, 60000)
inflow_sampler = qewton.RandomUniformSampler(inflow_goe, 8000)
outflow_sampler = qewton.RandomUniformSampler(outflow_geo, 8000, compute_normals=True, normal_name=N)
side_sampler = qewton.RandomUniformSampler(side_geo, 15000)

In [9]:
# Point sampling:
import plotly.io as pio
pio.renderers.default = "vscode"

inflow_plot = inflow_sampler.visualize().plots[0]
outflow_plot = outflow_sampler.visualize().plots[0]
side_plot = side_sampler.visualize().plots[0]

fig = qewton.visualization.Figure(
    qewton.visualization.Overlay(inflow_plot, outflow_plot, side_plot)
)
app = qewton.visualization.DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [10]:
model = qewton.FCN(
    in_neurons=X,
    hidden_neurons=40,
    out_neurons=U+P,
    n_hidden_layers=6,
    activation=qewton.bb.Tanh,
)

In [11]:
def momentum_residual(u: U, p: P, x: X):  # type: ignore
    return mu * u.sym_grad(x).matrix_div(x) - p.gradient(x)

momentum_graph = qewton.PINNPipeline(volume_sampler, [model], 
                                     residual=momentum_residual, 
                                     residual_name="MomentumConstraint", 
                                     weight=100.0)
momentum_constraint = momentum_graph.constraint

In [12]:
def mass_residual(u: U, x: X):  # type: ignore
    return u.div(x)

mass_graph = qewton.PINNPipeline(volume_sampler, [model], 
                                 residual=mass_residual, 
                                 residual_name="MassConstraint", 
                                 weight=500.0)
mass_constraint = mass_graph.constraint

In [13]:
def inflow_residual(u: U, x: X):  # type: ignore
    inflow_function = qewton.bb.ZerosLike()(x)
    inflow_function[:, 1] = - ((x[:, 0]-1.0)**2 + x[:, 2]**2 - 0.01)/0.01
    return u - inflow_function

in_graph = qewton.PINNPipeline(inflow_sampler, [model], 
                                residual=inflow_residual, 
                                residual_name="InConstraint", 
                                weight=1.0)
in_constraint = in_graph.constraint

In [14]:
def no_slip_residual(u: U):  # type: ignore
    return u

no_slip_graph = qewton.PINNPipeline(side_sampler, [model], 
                                    residual=no_slip_residual, 
                                    residual_name="SlipConstraint", 
                                    weight=100.0)
no_slip_constraint = no_slip_graph.constraint

In [15]:
def outflow_residual(p: P):  # type: ignore
    return p

out_graph = qewton.PINNPipeline(outflow_sampler, [model], 
                                residual=outflow_residual, 
                                residual_name="OutConstraint", 
                                weight=1.0)
out_constraint = out_graph.constraint

In [ ]:
# Train inflow boundary condition first:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=100,
)


trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[in_graph],
    training_objectives=[in_constraint],
    device=qewton.cuda(1),
)

trainer.run()

In [ ]:
# Train all other boundary condition:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.0005,
    max_iterations=10000,
)


trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[in_graph, out_graph, no_slip_graph],
    training_objectives=[in_constraint, out_constraint, no_slip_constraint],
    device=qewton.cuda(1),
)

trainer.run()

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.0001,
    max_iterations=5000,
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    lr=0.1,
    max_iterations=1000,
    optimizer_args={"max_eval": 20},
)

fix_sampler_callback = qewton.optim.CacheDataCallback(
    data_nodes=[volume_sampler, inflow_sampler, side_sampler, outflow_sampler],
    phase_to_start_cache=lbfgs_phase
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase, lbfgs_phase],
    graphs=[momentum_graph, mass_graph, in_graph, out_graph, no_slip_graph],
    callbacks=[fix_sampler_callback],
    training_objectives=[momentum_constraint, mass_constraint, in_constraint, 
                         out_constraint, no_slip_constraint],
    device=qewton.cuda(1),
)

trainer.run()

In [ ]:
import numpy as np

# new visualization way:

fem_velocity = np.load("./data/stokes_pipe/velocity.npy")
mesh_coords = np.load("./data/stokes_pipe/mesh_coords.npy").astype(np.float32)

fem_geometry = qewton.geometries.PointCloud(X, mesh_coords)
velocity_config = qewton.DataConfiguration(qewton.GeometryAxes(fem_geometry), qewton.FeatureAxes(U))


fig = qewton.viz.Figure(
    momentum_graph.visualize(model.output_ports[0],
                             variables=[U],
                             reference=fem_velocity,
                             reference_config=velocity_config,
                             )
    )
# could be improved by using colors etc, but for now its working

app = qewton.viz.DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [ ]:
import torch
import numpy as np

fem_velocity = np.load("./data/stokes_pipe/velocity.npy")
mesh_coords = np.load("./data/stokes_pipe/mesh_coords.npy").astype(np.float32)

model.to("cpu")
model_out = model(torch.tensor(mesh_coords))[:, :3].detach().cpu().numpy()

fem_geometry = qewton.geometries.PointCloud(X, mesh_coords)
velocity_config = qewton.DataConfiguration(qewton.GeometryAxes(fem_geometry), qewton.FeatureAxes(U))

reference_plot = qewton.visualization.auto_plot(fem_velocity, velocity_config, title="FEM Reference")
prediction_plot = qewton.visualization.auto_plot(model_out, velocity_config, title="PINN Prediction")

error_point_wise = np.linalg.norm(model_out - fem_velocity, ord=2, axis=1, keepdims=True).astype(np.float32)
ERROR = qewton.Variable("error", 1)
error_config = qewton.DataConfiguration(qewton.GeometryAxes(fem_geometry), qewton.FeatureAxes(ERROR))
error_plot = qewton.visualization.auto_plot(error_point_wise, error_config, title="Point-wise Error")

fig = qewton.visualization.Figure(
    qewton.visualization.Row(reference_plot, prediction_plot, error_plot)
)
app = qewton.viz.DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")

velocity_l2 = np.linalg.norm(fem_velocity, ord=2, axis=1)
mse_error = np.mean(error_point_wise[:, 0] ** 2)
print("MSE between PINN and FEM:", mse_error)
print("Relative MSE:", np.mean((error_point_wise[:, 0] / (velocity_l2 + 1.e-2)) ** 2))

Dash app running on http://127.0.0.1:8050/


In [ ]:
inflow_sampler = qewton.RandomUniformSampler(inflow_goe, 1000)
p = inflow_sampler()
u = model(p)[:, :3]
u_in = torch.zeros_like(p)
u_in[:, 1] = - ((p[:, 0]-1.0)**2 + p[:, 2]**2 - 0.01)/0.01

speed = torch.linalg.norm(u, axis=1).detach().cpu().numpy().reshape(-1, 1).astype(np.float32)
SPEED = qewton.Variable("speed", 1)
inflow_projection = qewton.geometries.PointCloud(
    qewton.Variable("x_proj", 2), p[:, [0, 2]].detach().cpu().numpy()
)
speed_config = qewton.DataConfiguration(
    qewton.GeometryAxes(inflow_projection), qewton.FeatureAxes(SPEED)
)
qewton.visualization.Figure(
    qewton.visualization.auto_plot(speed, speed_config, title="Inflow speed")
).show()